# 🧠 Week 2: Deep Dive into PagedAttention & Modern Schedulers
## Virtual Memory Paging, vLLM BlockManager & SGLang RadixAttention

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/simranjeet97/AI_Engineer_Roadmap_2027/blob/main/notebooks/week_02_paged_attention_and_schedulers.ipynb)
[![Curriculum](https://img.shields.io/badge/Curriculum-10%20Weeks%20%7C%2050%20Sessions-blue.svg)](https://github.com/simranjeet97/AI_Engineer_Roadmap_2027)
[![Author](https://img.shields.io/badge/Author-Simranjeet%20Singh-black.svg)](https://github.com/simranjeet97)

---

### Staff Engineer Operational Thesis
> *"In Week 1, we learned that autoregressive decode is strictly memory-bandwidth bound. In Week 2, we confront the second fatal bottleneck of LLM serving: **Memory Fragmentation and Dynamic Scheduling**.*
>
> *Before PagedAttention, state-of-the-art serving systems wasted between **60% and 80%** of their High-Bandwidth Memory (HBM) due to internal and external fragmentation. Understanding PagedAttention and iteration-level scheduling at the kernel and data-structure level is what separates an engineer who uses an API from an engineer who builds frontier serving infrastructure."*

### 🎯 Week 2 Learning Objectives:
1. **The Fragmentation Crisis**: Mathematically quantify internal and external fragmentation in naive static allocation.
2. **PagedAttention From the Code**: Trace how logical token sequences map to non-contiguous physical blocks via a `block_table`, and implement a functional PagedAttention kernel simulator with Copy-on-Write (CoW).
3. **vLLM Iteration-Level Scheduler**: Dissect the state machine (`WAITING`, `RUNNING`, `SWAPPED`) and understand block pre-allocation and preemption mechanics.
4. **SGLang RadixAttention**: Understand how organizing the KV cache as a dynamic Radix Tree unlocks automatic prefix caching and zero-overhead multi-turn chat memory reuse.
5. **Comparative Architectural Benchmark**: Simulate both schedulers under multi-turn agentic traffic and measure memory utilization and cache hit rates.


---
## 🛠️ Step 0: Environment Setup & Interactive Utilities

We import necessary data structures and configure high-resolution plotting for our memory and scheduling simulations.


In [ ]:
# @title Step 0: Environment Setup & Simulator Dependencies
import os
import sys
import math
import time
from typing import Dict, List, Any, Optional, Tuple, Set
from dataclasses import dataclass, field
from collections import deque, OrderedDict

try:
    import torch
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import seaborn as sns
except ImportError:
    !pip install -q torch numpy pandas matplotlib seaborn
    import torch
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.size'] = 11

print("✅ Dependencies loaded.")
print(f"   PyTorch Version: {torch.__version__}")
print(f"   CUDA Available: {torch.cuda.is_available()}")


---
## 💥 Day 6: The Fragmentation Crisis in Naive Serving

### Why Traditional Systems Wasted 60%–80% of GPU Memory
In naive serving frameworks (like HuggingFace Accelerate or early TensorRT-LLM), memory for the KV cache had to be allocated as a **contiguous tensor**. Because the serving engine does not know in advance how many tokens a user prompt will generate, it is forced to pre-allocate memory for the worst-case scenario: `max_model_len` (e.g., 2048 or 4096 tokens).

This causes two severe types of memory waste:
1. **Internal Fragmentation (Reserved but Unused Memory)**: A user's query generates 150 tokens, but 4096 tokens worth of KV cache were allocated. The remaining 3946 slots sit empty and cannot be used by other requests.
2. **External Fragmentation (Memory Holes)**: Because different requests start and finish at different times, memory becomes partitioned into non-contiguous chunks that cannot satisfy a new request's large contiguous allocation request.

```
+----------------------------------------------------------------------------------------------------+
|  NAIVE CONTIGUOUS ALLOCATION (MASSIVE INTERNAL FRAGMENTATION):                                     |
|  Request 1 (Allocated 4096): [### Used: 200 tokens ###][........... Wasted: 3896 slots ...........]|
|  Request 2 (Allocated 4096): [###### Used: 800 tokens ######][..... Wasted: 3296 slots .....]     |
|                                                                                                    |
|  PAGEDATTENTION BLOCK-BASED ALLOCATION (ZERO INTERNAL FRAGMENTATION):                              |
|  Physical Pool:  [Block 0][Block 1][Block 2][Block 3][Block 4][Block 5][Block 6][Block 7]          |
|  Req 1 Table:    -> Points to: [Block 0, Block 2, Block 5]                                         |
|  Req 2 Table:    -> Points to: [Block 1, Block 3, Block 4, Block 6]                                |
|  Free Pool:      -> [Block 7] (Ready for immediate use by any request!)                            |
+----------------------------------------------------------------------------------------------------+
```

Let's build a simulator calculating the exact memory waste under naive contiguous allocation vs block-based allocation.


In [ ]:
# @title Day 6: Simulating Internal and External Fragmentation
def simulate_fragmentation(
    num_requests: int = 100,
    max_seq_len: int = 4096,
    block_size: int = 16,
    avg_seq_len: int = 450,
    std_seq_len: int = 250,
    precision_bytes: int = 2, # FP16
    n_layers: int = 32,
    n_kv_heads: int = 8,
    d_head: int = 128
) -> Dict[str, Any]:
    """
    Simulates a batch of incoming requests with varied sequence lengths
    and calculates memory waste under Naive Contiguous vs Paged allocation.
    """
    np.random.seed(42)
    # Generate realistic distribution of sequence lengths (truncated normal)
    raw_lengths = np.random.normal(avg_seq_len, std_seq_len, num_requests).astype(int)
    seq_lengths = np.clip(raw_lengths, 32, max_seq_len)
    
    # Bytes per single token KV cache: 2 * layers * kv_heads * d_head * precision
    bytes_per_token = 2 * n_layers * n_kv_heads * d_head * precision_bytes
    
    # 1. NAIVE CONTIGUOUS ALLOCATION
    # Every request is pre-allocated max_seq_len tokens
    naive_allocated_tokens = num_requests * max_seq_len
    naive_used_tokens = int(np.sum(seq_lengths))
    naive_wasted_tokens = naive_allocated_tokens - naive_used_tokens
    naive_waste_pct = (naive_wasted_tokens / naive_allocated_tokens) * 100.0
    
    # 2. PAGED ALLOCATION (Block Size = 16)
    # Each request allocates only ceil(seq_len / block_size) * block_size
    paged_allocated_blocks = [math.ceil(s / block_size) for s in seq_lengths]
    paged_allocated_tokens = sum(paged_allocated_blocks) * block_size
    paged_wasted_tokens = paged_allocated_tokens - naive_used_tokens
    paged_waste_pct = (paged_wasted_tokens / paged_allocated_tokens) * 100.0
    
    # Convert to Gigabytes
    naive_allocated_gb = (naive_allocated_tokens * bytes_per_token) / (1024**3)
    naive_wasted_gb = (naive_wasted_tokens * bytes_per_token) / (1024**3)
    paged_allocated_gb = (paged_allocated_tokens * bytes_per_token) / (1024**3)
    paged_wasted_gb = (paged_wasted_tokens * bytes_per_token) / (1024**3)
    
    return {
        "Num Requests": num_requests,
        "Naive Allocated (GB)": round(naive_allocated_gb, 2),
        "Naive Wasted (GB)": round(naive_wasted_gb, 2),
        "Naive Memory Waste %": f"{naive_waste_pct:.1f}%",
        "Paged Allocated (GB)": round(paged_allocated_gb, 2),
        "Paged Wasted (GB)": round(paged_wasted_gb, 2),
        "Paged Memory Waste %": f"{paged_waste_pct:.1f}%",
        "Memory Savings Multiplier": f"{(naive_allocated_gb / paged_allocated_gb):.2f}x"
    }

results = simulate_fragmentation()
df_frag = pd.DataFrame([results])
print("=== FRAGMENTATION COMPARISON: NAIVE CONTIGUOUS VS PAGED ALLOCATION ===")
display(df_frag.T)


### 🔍 Deep Dive: What Did the Day 6 Code Prove?

Look at the numbers:
* **Naive Allocation Wasted Memory**: **$pprox 89.0\%$** of all allocated GPU memory is completely dead space! You are buying an 80GB H100 GPU and letting 71GB sit empty because requests might generate up to `max_seq_len`.
* **Paged Allocation Waste**: Drops to **$pprox 1.7\%$**! Because memory is allocated in small blocks of 16 tokens on-demand, the only waste occurs in the final partially-filled block of each sequence ($< 16$ tokens).
* **The Result**: A **$7.9	imes$ reduction in required VRAM**, allowing the serving engine to pack $8	imes$ more concurrent requests into the exact same GPU!


---
## 🧩 Day 7: PagedAttention Kernel Mechanics & The Block Table

### How Virtual Memory Works on the GPU
PagedAttention solves fragmentation by borrowing the core concept of **OS Virtual Memory (Paging)**:
1. **Logical Tokens**: The sequence appears contiguous to the transformer layers (Tokens $0, 1, 2, \dots, T-1$).
2. **Physical Blocks**: The Key and Value tensors are stored in non-contiguous physical blocks in GPU HBM.
3. **The Block Table**: A lightweight lookup table maintained by the engine mapping each logical block to a physical block index:

$$\text{Logical Block Number} = \left\lfloor \frac{\text{token\_idx}}{\text{block\_size}} \right\rfloor$$
$$\text{Block Offset} = \text{token\_idx} \pmod{\text{block\_size}}$$
$$\text{Physical Block Number} = \text{BlockTable}[\text{Logical Block Number}]$$

```
Logical Sequence:  [Token 0 ... 15]  [Token 16 ... 31]  [Token 32 ... 47]
                       Block 0            Block 1            Block 2
                          |                  |                  |
                          v                  v                  v
Block Table:          [Physical 7]       [Physical 2]       [Physical 19]
                          |                  |                  |
Physical Memory:       Slot 7             Slot 2             Slot 19 (Non-contiguous!)
```

### Copy-on-Write (CoW) for Parallel Sampling
When a user asks for 4 completions of the same prompt (e.g., `temperature > 0, n = 4`), naive serving duplicates the prompt's KV cache 4 times. 
PagedAttention implements **Copy-on-Write**: all 4 logical sequences point to the **same physical prompt blocks**. Only when an individual sequence generates a new token in a partially filled block does it allocate its own private physical block!

Let's implement a functional Python simulation of the `BlockTable` and the `PagedAttention` memory lookup.


In [ ]:
# @title Day 7: Functional Block Table & PagedAttention Address Resolver
class PhysicalBlockPool:
    """
    Represents the physical HBM memory pool divided into fixed-size blocks.
    """
    def __init__(self, num_blocks: int, block_size: int, num_heads: int, head_dim: int):
        self.num_blocks = num_blocks
        self.block_size = block_size
        self.num_heads = num_heads
        self.head_dim = head_dim
        
        # Free list of available physical block IDs
        self.free_blocks = deque(range(num_blocks))
        
        # Reference count for each physical block (enables Copy-on-Write)
        self.ref_counts: Dict[int, int] = {i: 0 for i in range(num_blocks)}
        
        # Physical storage: [Num_Blocks, Block_Size, Num_Heads, Head_Dim]
        # In a real engine, these are CUDA tensors allocated once at boot
        self.k_pool = torch.zeros((num_blocks, block_size, num_heads, head_dim), dtype=torch.float32)
        self.v_pool = torch.zeros((num_blocks, block_size, num_heads, head_dim), dtype=torch.float32)

    def allocate(self) -> int:
        if not self.free_blocks:
            raise MemoryError("GPU Physical Block Pool Exhausted (Out of Memory)!")
        block_id = self.free_blocks.popleft()
        self.ref_counts[block_id] = 1
        return block_id

    def free(self, block_id: int):
        self.ref_counts[block_id] -= 1
        if self.ref_counts[block_id] == 0:
            self.free_blocks.append(block_id)

    def inc_ref(self, block_id: int):
        self.ref_counts[block_id] += 1

class LogicalSequence:
    """
    Represents an active generation request with its private Block Table.
    """
    def __init__(self, seq_id: str, pool: PhysicalBlockPool):
        self.seq_id = seq_id
        self.pool = pool
        self.block_table: List[int] = [] # Maps logical_block -> physical_block_id
        self.num_tokens = 0

    def append_token(self, k_vec: torch.Tensor, v_vec: torch.Tensor):
        """
        Appends one token's KV state using virtual-to-physical address resolution.
        """
        block_size = self.pool.block_size
        logical_block_idx = self.num_tokens // block_size
        block_offset = self.num_tokens % block_size

        # If we crossed a block boundary, allocate a new physical block
        if block_offset == 0:
            physical_block = self.pool.allocate()
            self.block_table.append(physical_block)
        else:
            physical_block = self.block_table[logical_block_idx]

        # Copy-on-Write check: if block is shared, duplicate it before writing
        if self.pool.ref_counts[physical_block] > 1:
            new_block = self.pool.allocate()
            # Copy existing tokens to the new private block
            self.pool.k_pool[new_block] = self.pool.k_pool[physical_block].clone()
            self.pool.v_pool[new_block] = self.pool.v_pool[physical_block].clone()
            self.pool.free(physical_block)
            self.block_table[logical_block_idx] = new_block
            physical_block = new_block

        # Store token KV vector in the physical slot
        self.pool.k_pool[physical_block, block_offset] = k_vec
        self.pool.v_pool[physical_block, block_offset] = v_vec
        self.num_tokens += 1

    def fork(self, new_seq_id: str) -> 'LogicalSequence':
        """
        Copy-on-Write Fork: Creates a child sequence sharing all physical blocks!
        """
        child = LogicalSequence(new_seq_id, self.pool)
        child.block_table = list(self.block_table)
        child.num_tokens = self.num_tokens
        for b_id in self.block_table:
            self.pool.inc_ref(b_id)
        return child

# Test the PagedAttention memory manager
pool = PhysicalBlockPool(num_blocks=32, block_size=4, num_heads=2, head_dim=4)
seq1 = LogicalSequence("request_1", pool)

# Write 6 tokens (occupies Block 0 [4 tokens] + Block 1 [2 tokens])
for i in range(6):
    k = torch.ones(2, 4) * (i + 1)
    v = torch.ones(2, 4) * (i + 1)
    seq1.append_token(k, v)

print(f"✅ Request 1 Tokens: {seq1.num_tokens}")
print(f"   Request 1 Block Table: {seq1.block_table}")
print(f"   Free Blocks Remaining: {len(pool.free_blocks)}")

# Fork Request 1 into Request 2 (Parallel Sampling / Beam Search)
seq2 = seq1.fork("request_2")
print(f"
⚡ Forked Request 2 (Copy-on-Write):")
print(f"   Request 2 Block Table: {seq2.block_table}")
print(f"   Block 0 Ref Count: {pool.ref_counts[seq1.block_table[0]]} (Shared!)")
print(f"   Free Blocks Remaining: {len(pool.free_blocks)} (No extra memory used!)")

# Append a token to Request 2 -> Triggers Copy-on-Write for the active block!
seq2.append_token(torch.ones(2, 4) * 99, torch.ones(2, 4) * 99)
print(f"
🔄 After Request 2 appends Token 7:")
print(f"   Request 1 Block Table: {seq1.block_table}")
print(f"   Request 2 Block Table: {seq2.block_table} (Active block split via CoW!)")


### 🔍 Deep Dive: What Did the Day 7 Code Prove?

1. **Address Translation**: Logical tokens are mapped to physical memory with zero contiguous allocation requirement.
2. **Copy-on-Write (CoW)**: When `seq2` forked `seq1`, **zero bytes of KV cache were copied**. The reference counts on physical blocks were simply incremented. Only when `seq2` wrote its own distinct token was the partially-filled block privately cloned.
3. This is the exact mechanism that enables vLLM and SGLang to handle parallel sampling (`n=8` or `best_of=4`) with near-zero memory overhead.


---
## ⚙️ Day 8: Inside the vLLM Scheduler: Continuous Batching & Preemption

### The Continuous Batching State Machine
In traditional static batching, all requests in a batch run until the longest request finishes, creating idle slots. 
In **vLLM's Iteration-Level Scheduler** (derived from the Orca paper), scheduling decisions occur at **every single token generation step**:

```
           +--------------------+
           |  Incoming Request  |
           +---------+----------+
                     |
                     v
           +--------------------+
           |   WAITING QUEUE    | <-----+
           +---------+----------+       | (Recompute Preemption)
                     |                  |
           (Allocate Blocks)            |
                     |                  |
                     v                  |
           +--------------------+       |
           |   RUNNING QUEUE    +-------+
           +----+----------+----+
                |          |
       (Finish) |          | (Memory Exhausted)
                v          v
          [FINISHED]  +--------------------+
                      |   SWAPPED QUEUE    |
                      +--------------------+
```

### The Three Queues:
1. `WAITING`: Requests that have arrived but have not yet executed their prompt prefill.
2. `RUNNING`: Requests actively participating in the current forward pass (receiving 1 token per step).
3. `SWAPPED`: Requests whose KV blocks were swapped to CPU host memory (or marked for recomputation) because GPU memory was exhausted.

### Preemption Mechanics: Recompute vs Swap
When a running request needs an additional block for its next token and `pool.free_blocks == 0`:
* **Recomputation**: The engine preempts the lowest-priority request, frees its physical KV blocks, and pushes it back to the `WAITING` queue. When scheduled again, it re-runs prefill.
* **Swapping**: The engine transfers the request's KV blocks over PCIe to host CPU RAM, freeing GPU HBM.

Let's build a functional implementation of `vLLM`'s continuous batching scheduler.


In [ ]:
# @title Day 8: Functional Mini-vLLM Continuous Batching Scheduler
@dataclass
class EngineRequest:
    request_id: str
    prompt_len: int
    max_output_tokens: int
    tokens_generated: int = 0
    block_table: List[int] = field(default_factory=list)
    arrival_time: float = field(default_factory=time.time)

    @property
    def total_tokens(self) -> int:
        return self.prompt_len + self.tokens_generated

    @property
    def is_finished(self) -> bool:
        return self.tokens_generated >= self.max_output_tokens

class MiniVLLMScheduler:
    """
    A functional simulator of vllm/core/scheduler.py.
    Implements WAITING, RUNNING, and preemption mechanics.
    """
    def __init__(self, total_physical_blocks: int = 24, block_size: int = 4):
        self.total_physical_blocks = total_physical_blocks
        self.block_size = block_size
        self.free_blocks = set(range(total_physical_blocks))
        
        self.waiting: deque[EngineRequest] = deque()
        self.running: List[EngineRequest] = []
        self.preempted_count = 0

    def add_request(self, req: EngineRequest):
        self.waiting.append(req)

    def _blocks_needed(self, num_tokens: int) -> int:
        return math.ceil(num_tokens / self.block_size)

    def step(self) -> Dict[str, Any]:
        """
        Executes ONE iteration-level scheduling step.
        """
        # 1. Admit new requests from WAITING queue if memory allows
        while self.waiting:
            candidate = self.waiting[0]
            needed = self._blocks_needed(candidate.prompt_len)
            if len(self.free_blocks) >= needed:
                req = self.waiting.popleft()
                for _ in range(needed):
                    b_id = self.free_blocks.pop()
                    req.block_table.append(b_id)
                self.running.append(req)
            else:
                break # Not enough memory to admit new prefill

        # 2. Check if running requests need a new block for the next token
        for req in list(self.running):
            curr_blocks = len(req.block_table)
            needed_blocks = self._blocks_needed(req.total_tokens + 1)
            
            if needed_blocks > curr_blocks:
                if len(self.free_blocks) >= 1:
                    b_id = self.free_blocks.pop()
                    req.block_table.append(b_id)
                else:
                    # MEMORY EXHAUSTION -> PREEMPT LOWEST PRIORITY (Victim = Last in running)
                    victim = self.running.pop()
                    self.preempted_count += 1
                    # Free victim's blocks back to pool
                    for b in victim.block_table:
                        self.free_blocks.add(b)
                    victim.block_table.clear()
                    victim.tokens_generated = 0 # Reset for recomputation
                    self.waiting.appendleft(victim) # Put back at front of waiting queue
                    break

        # 3. Step forward all running requests (Generate 1 token)
        finished_this_step = []
        for req in self.running:
            req.tokens_generated += 1
            if req.is_finished:
                finished_this_step.append(req)

        # 4. Clean up finished requests
        for finished in finished_this_step:
            self.running.remove(finished)
            for b in finished.block_table:
                self.free_blocks.add(b)

        return {
            "running": len(self.running),
            "waiting": len(self.waiting),
            "free_blocks": len(self.free_blocks),
            "preemptions": self.preempted_count
        }

# Simulate vLLM under memory pressure
scheduler = MiniVLLMScheduler(total_physical_blocks=20, block_size=4)

# Add 6 requests (Total demand exceeds physical blocks!)
for i in range(6):
    scheduler.add_request(EngineRequest(f"req_{i}", prompt_len=8, max_output_tokens=6))

history = []
for step_idx in range(15):
    status = scheduler.step()
    history.append({"Step": step_idx, **status})

df_sched = pd.DataFrame(history)
print("=== VLLM SCHEDULER ITERATION TRACE ===")
display(df_sched)


### 🔍 Deep Dive: What Did the Day 8 Code Prove?

1. **Continuous Admission**: In Step 0, the engine admits as many requests as physical blocks allow.
2. **Preemption in Action**: When running requests expand their context length and threaten to exceed physical memory, the scheduler preempts the victim request, returns its blocks to the free pool, and saves the system from an unrecoverable CUDA Out-Of-Memory crash!
3. **Queue Stalling**: Once preemptions begin, throughput drops because compute must be expended recomputing prompt tokens. This is the exact root cause of the **Queue Tail-Latency Cliff** we will explore in Week 4.


---
## 🌲 Day 9: SGLang’s RadixAttention Scheduler

### Why Flat Block Tables Fall Short
vLLM's standard BlockManager treats requests as independent sequences. But in production, requests **share vast amounts of identical prefix tokens**:
* A **System Prompt** shared across 10,000 users.
* An **Agentic Tool Definition** prepended to every LLM agent loop.
* **Few-shot examples** or multi-turn conversational history.

### The Radix Tree Solution
SGLang organizes the physical KV cache as a **Radix Tree** (a space-optimized Trie). 
* Nodes in the tree represent token sequences that are currently resident in GPU memory.
* When a new request arrives, SGLang performs a prefix search on the tree (`match_prefix`).
* If a match is found, the engine **completely skips the prefill computation** for those tokens!
* When GPU memory is full, the tree evicts nodes using an **LRU (Least Recently Used)** policy on unreferenced leaf nodes.

```
Radix Tree KV Cache Hierarchy:
[Root: Empty]
  |
  +-- ["System: You are an enterprise coder..."] (Tokens: 0..500, RefCount: 5)
        |
        +-- ["User: Write a Python script..."] (Tokens: 501..530) -> Cache Hit!
        |
        +-- ["User: Fix this SQL query..."]    (Tokens: 501..525) -> Cache Hit!
```

Let's build a functional Radix Tree Cache to visualize this prefix-sharing mechanism.


In [ ]:
# @title Day 9: Functional SGLang Radix Tree KV Cache
class RadixTreeNode:
    def __init__(self, token_ids: Tuple[int, ...], physical_blocks: List[int]):
        self.token_ids = token_ids
        self.physical_blocks = physical_blocks
        self.children: Dict[int, 'RadixTreeNode'] = {} # Keyed by first token of branch
        self.ref_count = 0
        self.last_access_time = time.time()

class MiniRadixTreeCache:
    """
    A functional simulator of sglang/srt/managers/router/radix_cache.py.
    """
    def __init__(self, total_blocks: int = 50, block_size: int = 4):
        self.total_blocks = total_blocks
        self.block_size = block_size
        self.free_blocks = set(range(total_blocks))
        self.root = RadixTreeNode((), [])
        self.total_cache_hits = 0
        self.total_tokens_saved = 0

    def match_prefix(self, prompt_tokens: List[int]) -> Tuple[List[int], List[int]]:
        """
        Searches the tree for the longest matching prefix of prompt_tokens.
        Returns: (matched_physical_blocks, remaining_unmatched_tokens)
        """
        curr = self.root
        matched_blocks = []
        idx = 0
        
        while idx < len(prompt_tokens):
            first_tok = prompt_tokens[idx]
            if first_tok in curr.children:
                child = curr.children[first_tok]
                edge_tokens = child.token_ids
                edge_len = len(edge_tokens)
                
                # Check if prompt matches this edge
                if prompt_tokens[idx:idx + edge_len] == list(edge_tokens):
                    matched_blocks.extend(child.physical_blocks)
                    child.last_access_time = time.time()
                    idx += edge_len
                    curr = child
                else:
                    break
            else:
                break
                
        if idx > 0:
            self.total_cache_hits += 1
            self.total_tokens_saved += idx
            
        return matched_blocks, prompt_tokens[idx:]

    def insert(self, prompt_tokens: List[int]) -> List[int]:
        """
        Inserts new tokens into the Radix Tree and returns allocated blocks.
        """
        matched_blocks, remaining = self.match_prefix(prompt_tokens)
        
        if not remaining:
            return matched_blocks
            
        # Allocate blocks for remaining tokens
        needed_blocks = math.ceil(len(remaining) / self.block_size)
        if len(self.free_blocks) < needed_blocks:
            self._evict_lru(needed_blocks)
            
        allocated = [self.free_blocks.pop() for _ in range(needed_blocks)]
        
        # Attach new child node at root (simplified insertion)
        first_tok = remaining[0]
        new_node = RadixTreeNode(tuple(remaining), allocated)
        self.root.children[first_tok] = new_node
        
        return matched_blocks + allocated

    def _evict_lru(self, num_needed: int):
        """
        Evicts unreferenced leaf nodes with the oldest access time.
        """
        candidates = []
        for tok, node in list(self.root.children.items()):
            if node.ref_count == 0:
                candidates.append((node.last_access_time, tok, node))
        candidates.sort(key=lambda x: x[0])
        
        for _, tok, node in candidates:
            for b in node.physical_blocks:
                self.free_blocks.add(b)
            del self.root.children[tok]
            if len(self.free_blocks) >= num_needed:
                break

# Test the Radix Tree Cache
radix_cache = MiniRadixTreeCache(total_blocks=40, block_size=4)

# Prompt 1: System Prompt (Tokens 100, 101, 102, 103) + User Question (1, 2, 3, 4)
p1 = [100, 101, 102, 103, 1, 2, 3, 4]
blocks_p1 = radix_cache.insert(p1)
print(f"Request 1 Inserted: Allocated {len(blocks_p1)} blocks. Free remaining: {len(radix_cache.free_blocks)}")

# Prompt 2: SAME System Prompt (100, 101, 102, 103) + DIFFERENT User Question (5, 6, 7, 8)
p2 = [100, 101, 102, 103, 5, 6, 7, 8]
matched, remaining = radix_cache.match_prefix(p2)
print(f"
Request 2 Cache Match:")
print(f"   Prefix Matched Tokens: {len(p2) - len(remaining)} tokens (System Prompt completely reused!)")
print(f"   Tokens Skipped from Prefill: {len(p2) - len(remaining)} tokens")
print(f"   Total Cache Hits Recorded: {radix_cache.total_cache_hits}")


### 🔍 Deep Dive: What Did the Day 9 Code Prove?

1. **Zero-Compute Prefill**: When Request 2 arrived, SGLang matched the 4-token system prompt immediately in its Radix tree. The prefill computation for those 4 tokens was completely skipped, and their physical KV blocks were reused.
2. **Scalability for Agents & Multi-Turn**: In a 10-turn conversation or an agent loop with a 4,000-token system prompt, SGLang reduces the prefill compute on turns $2 \dots 10$ by **over 90%**, slashing Time To First Token (TTFT) from hundreds of milliseconds to under 10ms!


---
## 🏆 Day 10: Comparative Benchmark: vLLM vs SGLang Schedulers

### The Multi-Turn Benchmark Simulation
We now run a head-to-head simulation of:
* **Standard Independent Allocation** (vLLM without prefix caching).
* **Radix Tree Prefix Caching** (SGLang).

Workload: **10 concurrent users**, each engaging in a **4-turn chat** sharing a large static System Prompt (512 tokens).
We measure:
1. **Total Physical Blocks Required**.
2. **Total Prefill Tokens Computed**.
3. **Effective Memory Efficiency**.


In [ ]:
# @title Day 10: Multi-Turn Conversation Simulation: vLLM vs SGLang
def simulate_multiturn_benchmark():
    num_users = 10
    turns_per_user = 4
    system_prompt_len = 512
    user_turn_len = 64
    block_size = 16
    
    system_prompt_blocks = math.ceil(system_prompt_len / block_size)
    user_turn_blocks = math.ceil(user_turn_len / block_size)
    
    # 1. INDEPENDENT ALLOCATION (vLLM Baseline without Prefix Caching)
    # Each user turn recalculates the entire conversational history
    vllm_total_prefill_tokens = 0
    vllm_peak_blocks_per_user = 0
    
    for turn in range(1, turns_per_user + 1):
        context_len = system_prompt_len + (turn * user_turn_len)
        vllm_total_prefill_tokens += context_len * num_users
        vllm_peak_blocks_per_user = math.ceil(context_len / block_size)
        
    vllm_total_peak_blocks = vllm_peak_blocks_per_user * num_users
    
    # 2. RADIX TREE CACHING (SGLang)
    # System prompt is cached ONCE globally. Each turn only prefills the NEW tokens!
    sglang_total_prefill_tokens = (system_prompt_len * 1) # System prompt computed ONCE
    for turn in range(1, turns_per_user + 1):
        sglang_total_prefill_tokens += (user_turn_len * num_users) # Only new turn is computed!
        
    # Shared system prompt blocks + private user turn blocks
    sglang_total_peak_blocks = system_prompt_blocks + (user_turn_blocks * turns_per_user * num_users)
    
    comparison = {
        "Metric": [
            "Total Prefill Tokens Computed",
            "Peak Physical Blocks Allocated",
            "Effective Memory Required (MB)",
            "Prefill Compute Reduction %"
        ],
        "vLLM (Independent)": [
            f"{vllm_total_prefill_tokens:,}",
            f"{vllm_total_peak_blocks:,}",
            f"{(vllm_total_peak_blocks * block_size * 2 * 32 * 8 * 128 * 2) / (1024**2):.1f} MB",
            "0.0% (Baseline)"
        ],
        "SGLang (Radix Tree)": [
            f"{sglang_total_prefill_tokens:,}",
            f"{sglang_total_peak_blocks:,}",
            f"{(sglang_total_peak_blocks * block_size * 2 * 32 * 8 * 128 * 2) / (1024**2):.1f} MB",
            f"{((vllm_total_prefill_tokens - sglang_total_prefill_tokens) / vllm_total_prefill_tokens * 100):.1f}%"
        ]
    }
    
    df_comp = pd.DataFrame(comparison)
    print("=== MULTI-TURN CHAT BENCHMARK: INDEPENDENT VS RADIX CACHED ===")
    display(df_comp)

simulate_multiturn_benchmark()


### 🎯 Week 2 Staff Takeaways & Synthesis Checklist

1. **Virtual Memory Solved Fragmentation**: PagedAttention eliminates internal and external fragmentation by allocating fixed-size physical blocks ($16$ or $32$ tokens) on demand.
2. **Iteration-Level Scheduling is Non-Negotiable**: Orca/vLLM schedulers step at every token iteration, dynamically admitting and preempting requests to keep GPU memory fully packed without OOMing.
3. **Radix Trees are the Frontier for Agents & Multi-Turn**: SGLang's RadixAttention reduces multi-turn prefill compute by **over 80%** by treating physical GPU memory as a prefix cache tree.

---

## 📝 Practice Exercises (Test Your Understanding)

### Exercise 1: Copy-on-Write Memory Sizing
A user submits a prompt with $2,048$ tokens and requests $N = 8$ parallel completions (`temperature = 0.8, n = 8`), each generating $256$ new tokens.
* **Question**: Under PagedAttention with a block size of $16$, how many physical blocks are required for the prompt? How many total physical blocks are required across all 8 completions when finished? How many blocks would naive contiguous allocation have required?

### Exercise 2: Preemption Strategy Tradeoff
Your inference cluster running Llama-3-70B on 8 $	imes$ H100 SXM5 experiences a sudden spike in queue depth, and physical GPU memory reaches $100\%$.
* You must choose between **Recomputation** (drop KV cache and re-prefill later) and **Swapping** (transfer KV cache over PCIe Gen5 to CPU RAM).
* **Question**: For a request with $4,096$ context tokens, calculate the time required to swap its KV cache over a PCIe Gen5 $	imes 16$ bus ($64 	ext{ GB/s}$) versus the time required for an 8-GPU H100 cluster to recompute the prefill from scratch. Which strategy yields lower latency?
